# Example: Apple 2020

<h1 align="center">
    <img src="https://raw.githubusercontent.com/reeyarn/openesef/refs/heads/master/markdown/esefdata.svg" alt="Open ESEF" style="max-width: 100%; height: auto;"/>
<br>A Python Library for ESEF and XBRL Filings
<br>
<img src="https://img.shields.io/badge/Project%20Status-Under%20Development-yellow" alt="Project Status: Under Development - 66% Complete" />
<img src="https://img.shields.io/badge/License-GPLv3-blue.svg" alt="License: GPL v3.0" />
</h1>


First, lets load the logger:

In [1]:
import logging
from openesef.util.util_mylogger import setup_logger 
logger = setup_logger("main", logging.CRITICAL, log_dir="/tmp/log/")

Now, lets load the xbrl filing, using Apple 2020 as an example:

In [2]:
from openesef.edgar.loader import load_xbrl_filing
#xid, tax = load_xbrl_filing(ticker="AAPL", year=2020)
xid, tax = load_xbrl_filing(filing_url="/Archives/edgar/data/320193/0000320193-20-000096.txt")

The xbrl filing is loaded and the taxonomy is created.

Now, lets print the XBRL instance (xid):

In [3]:
print(xid)


Namespaces: 10
Schema references: 1
Linkbase references: 0
Contexts: 318
Units: 9
Facts: 1388
Footnotes: 0
Filing Indicators: 0


And the taxonomy (tax):

In [4]:
print(tax)

Schemas: 14
Linkbases: 7
Role Types: 775
Arcrole Types: 0
Concepts: 18293
Item Types: 52
Tuple Types: 0
Simple Types: 0
Labels: 0
References: 0
Hierarchies: 79
Dimensional Relationship Sets: 79
Dimensions: 295
Hypercubes: 363
Enumerations: 0
Enumerations Sets: 0
Table Groups: 0
Tables: 0
Parameters: 0
Assertion Sets: 0
Value Assertions: 0
Existence Assertions: 0
Consistency Assertions: 0


DEI stands for Document and Entity Information. For each XBRL report, there will be a section for DEI,
and this class is to provide easy access to those commonly-defined DEI attributes.

Lets print the DEI:

In [5]:
for i, (key, value) in enumerate(xid.dei.items()):
    print(f"{i}: {key}: {value}")


0: AmendmentFlag: false
1: DocumentFiscalYearFocus: 2020
2: DocumentFiscalPeriodFocus: FY
3: EntityCentralIndexKey: 0000320193
4: CurrentFiscalYearEndDate: --09-26
5: DocumentType: 10-K
6: DocumentAnnualReport: true
7: DocumentPeriodEndDate: 2020-09-26
8: DocumentTransitionReport: false
9: EntityFileNumber: 001-36743
10: EntityRegistrantName: Apple Inc.
11: EntityIncorporationStateCountryCode: CA
12: EntityTaxIdentificationNumber: 94-2404110
13: EntityAddressAddressLine1: One Apple Park Way
14: EntityAddressCityOrTown: Cupertino
15: EntityAddressStateOrProvince: CA
16: EntityAddressPostalZipCode: 95014
17: CityAreaCode: 408
18: LocalPhoneNumber: 996-1010
19: TradingSymbol: AAPL
20: EntityWellKnownSeasonedIssuer: Yes
21: EntityVoluntaryFilers: No
22: EntityCurrentReportingStatus: Yes
23: EntityInteractiveDataCurrent: Yes
24: EntityFilerCategory: Large Accelerated Filer
25: EntitySmallBusiness: false
26: EntityEmergingGrowthCompany: false
27: IcfrAuditorAttestationFlag: true
28: EntitySh

In [6]:
from openesef.engines.tax_pres import TaxonomyPresentation
t_pres = TaxonomyPresentation(tax)


In [7]:
print("\nConcept Labels in Statement of Operations:")
concepts_statement_of_operations = []

for concept in t_pres.statement_concepts.values():
    if concept['statement_name'] == 'CONSOLIDATEDSTATEMENTSOFOPERATIONS':
        concepts_statement_of_operations.append(concept['concept_qname'])
        print("-"*30)
        print(f"Statement: {concept['statement_name']}")
        print(f"Concept: {concept['concept_qname']}")
        print(f"Label: {concept['label']}")        
    



Concept Labels in Statement of Operations:
------------------------------
Statement: CONSOLIDATEDSTATEMENTSOFOPERATIONS
Concept: us-gaap:IncomeStatementAbstract
Label: Income Statement [Abstract]
------------------------------
Statement: CONSOLIDATEDSTATEMENTSOFOPERATIONS
Concept: srt:ProductOrServiceAxis
Label: Product and Service [Axis]
------------------------------
Statement: CONSOLIDATEDSTATEMENTSOFOPERATIONS
Concept: srt:ProductsAndServicesDomain
Label: Product and Service [Domain]
------------------------------
Statement: CONSOLIDATEDSTATEMENTSOFOPERATIONS
Concept: us-gaap:ProductMember
Label: Product [Member]
------------------------------
Statement: CONSOLIDATEDSTATEMENTSOFOPERATIONS
Concept: us-gaap:ServiceMember
Label: Service [Member]
------------------------------
Statement: CONSOLIDATEDSTATEMENTSOFOPERATIONS
Concept: us-gaap:RevenueFromContractWithCustomerExcludingAssessedTax
Label: Revenue from Contract with Customer, Excluding Assessed Tax
-----------------------------

In [8]:
# Get the current year's main instance context
periods_dict = xid.identify_reporting_contexts()
#import pandas as pd
#print(pd.DataFrame.from_dict(periods_dict, orient='index'))


In [9]:
current_contexts = [ctx_id for ctx_id, ctx_info in periods_dict.items() 
                    if ctx_info['relative_year'] == 0 and  ctx_info.get('main_context')]

print(current_contexts)

['i747bec89b4e84f74ae3445db3509f609_I20200926', 'i223bd574caab4f739f73936be6065c72_D20190929-20200926', 'ic3ea678a3e394e00880d68882e8bdc02_I20200327', 'i5085fb79a9a14a9aae9b909beb32bce2_D20180930-20190928', 'i4920908218084be688c8fa96b8903033_D20171001-20180929']


In [10]:
print("\nFact Values:")
for key, fact in xid.xbrl.facts.items():
    concept_qname = fact.qname if hasattr(fact, 'qname') else 'N/A'  # Get the concept's QName
    context = xid.xbrl.contexts[fact.context_ref]
    period_info = periods_dict.get(fact.context_ref, {})
    period_string = period_info.get('period_string', 'N/A')
    if concept_qname in concepts_statement_of_operations and fact.context_ref in current_contexts:
        print(f"{concept_qname:<90} Value: {fact.value:<15} Context: {period_string}")    


Fact Values:
us-gaap:RevenueFromContractWithCustomerExcludingAssessedTax                                Value: 274515000000    Context: 2019-09-29/2020-09-26
us-gaap:RevenueFromContractWithCustomerExcludingAssessedTax                                Value: 260174000000    Context: 2018-09-30/2019-09-28
us-gaap:RevenueFromContractWithCustomerExcludingAssessedTax                                Value: 265595000000    Context: 2017-10-01/2018-09-29
us-gaap:CostOfGoodsAndServicesSold                                                         Value: 169559000000    Context: 2019-09-29/2020-09-26
us-gaap:CostOfGoodsAndServicesSold                                                         Value: 161782000000    Context: 2018-09-30/2019-09-28
us-gaap:CostOfGoodsAndServicesSold                                                         Value: 163756000000    Context: 2017-10-01/2018-09-29
us-gaap:GrossProfit                                                                        Value: 104956000000    Co